# Traceability — every number in the submitted manuscript → its artifact

Every number is read from an aggregate result artifact in `../artifacts/` — the files the submitted manuscript's tables were generated from.
No licensed microdata is used or required (see `../DATA_ACCESS.md`). Outputs are stored, so the notebook renders on GitHub without running.


## Table cells and prose numbers → sources
`EXHIBIT_MAP.csv` lists every numeric table cell; `TEXT_NUMBERS.csv` every number in the prose (References excluded).


In [ ]:
import csv, json, os
from collections import Counter
doc = "appendix"
em = [r for r in csv.DictReader(open("../EXHIBIT_MAP.csv", encoding="utf-8")) if r["document"] == doc]
tn = [r for r in csv.DictReader(open("../TEXT_NUMBERS.csv", encoding="utf-8")) if r["document"] == doc]
print(f"table cells mapped: {len(em)} · by source kind:", dict(Counter("ledger" if r["claim_id"] else ("constant" if "constant" in r["source"] else ("UNTRACEABLE" if r["source"]=="UNTRACEABLE" else "artifact")) for r in em)))
print(f"prose numbers: {len(tn)} · by kind:", dict(Counter(r["kind"] for r in tn)))
print("\nartifacts feeding this document's tables:", sorted({r["source"].split(":")[0] for r in em if ":" in r["source"]}))
print("pipeline scripts:", sorted({r["pipeline_script"] for r in em if r["pipeline_script"]}))
un = [r for r in em if r["source"] == "UNTRACEABLE"] + [r for r in tn if r["kind"] == "UNMATCHED"]
for r in un: print("  UNTRACEABLE:", r)


table cells mapped: 331 · by source kind: {'ledger': 203, 'artifact': 127, 'UNTRACEABLE': 1}
prose numbers: 308 · by kind: {'constant/count': 51, 'artifact': 87, 'artifact-abs': 2, 'ledger': 155, 'artifact-pct': 13}

artifacts feeding this document's tables: ['I04c.json', 'I05.json', 'I06.json', 'I11.json', 'I14.json', 'I19.json', 'I19c.json', 'I22.json', 'I25.json', 'I35.json', 'I36.json', 'I38.json', 'I39.json', 'I40.json', 'I41.json', 'I44.json', 'I45.json', 'I47.json', 'I48.json', 'I56.json', 'I57.json', 'I57_levels.json', 'I60.json', 'I61.json', 'I62.json', 'I63.json', 'I64.json', 'I65.json', 'I66.json', 'I67.json']
pipeline scripts: ['i04c_valueadded.py', 'i05_exit_reversal.py', 'i06_notyet_anatomy.py', 'i11_honestdid.py', 'i14_shareholder_dose.py', 'i19_succession.py', 'i19c_dose_gradient.py', 'i22_wage_structure.py', 'i25_pre_inertia.py', 'i35_canonical.py', 'i36_regression_table.py', 'i38_excess_zeros.py', 'i39_spell_benchmark.py', 'i40_salvage.py', 'i41_moderator_defense.py',

## The claims ledger resolves against the artifacts


In [ ]:
rows = list(csv.DictReader(open("../artifacts/CLAIMS_LEDGER.csv", encoding="utf-8-sig")))
def resolve(o, path):
    for k in [k for k in path.split(".") if k]: o = o[int(k)] if isinstance(o, list) else o[k]
    return o
exact = derived = mismatch = missing = 0
for r in rows:
    f = os.path.join("../artifacts", os.path.basename(r["source_json"]))
    if not os.path.exists(f): missing += 1; continue
    try: o = resolve(json.load(open(f, encoding="utf-8")), r["json_path"])
    except Exception: mismatch += 1; continue
    if isinstance(o, (dict, list)): derived += 1; continue
    try: ok = abs(float(r["value"]) - float(o)) <= max(5e-5, abs(float(o)) * 1e-6)
    except Exception: ok = str(o) == r["value"]
    exact += ok; mismatch += (not ok)
print(f"claims ledger: {len(rows)} rows · exact {exact} · derived {derived} · mismatch {mismatch} · artifact missing {missing}"); assert mismatch == 0


claims ledger: 351 rows · exact 345 · derived 6 · mismatch 0 · artifact missing 0
